In [ ]:
import pandas as pd
import numpy as np
import ast
import os 
import zipfile
from typing import List, Optional, Iterable, Dict, Any

def _normalize_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.str.replace(r"^(?!(chr))", "chr", regex=True)

def add_bins(df: pd.DataFrame, start_col: str, end_col: str, bin_size) -> pd.DataFrame:
    """Coarse binning for fast prefilter; exact distances still computed later."""
    out = df.copy()
    out["bin_start"] = (out[start_col] // bin_size).astype(np.int64)
    out["bin_end"]   = (out[end_col]   // bin_size).astype(np.int64)
    out["bins"] = [list(range(s, e + 1)) for s, e in zip(out["bin_start"], out["bin_end"])]
    return out

def _make_cut_edges_for_labels(base_edges, include_zero_bin=True):
    """
    Make edges so that len(labels) == len(edges)-1.
    With base_edges=[0,100k,250k,500k,1M] and 4 labels, we need edges[-0.1,100k,250k,500k,1M].
    """
    if include_zero_bin:
        return [-0.1] + base_edges[1:]
    else:
        return base_edges

def _as_categorical(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

def safe_parse_list(x):
    if pd.isna(x) or x in ["", "[]"]:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            # fall back: split comma-separated strings
            return [g for g in x.split(",") if g.strip() != ""]
    return []


def harmonize_chrom(genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, regulatory_elements: pd.DataFrame):
    was_changed = [False, False, False]
    for i, df in enumerate([genes, lncRNAs_genes, regulatory_elements]):
        print(i)
        if "seqname" in df.columns:
            df.rename(columns={"seqname": "chrom"}, inplace=True)
            was_changed[i] = True
        elif isinstance(df["chrom"].iloc[0], int):
            df["chrom"] = _normalize_chr(df["chrom"])
            was_changed[i] = True
    return genes, lncRNAs_genes, regulatory_elements, was_changed


def filter_selected_genes(genes: pd.DataFrame, primary_genes: list[str]) -> pd.DataFrame:
    genes = genes[genes["feature"] == "gene"]
    return genes[genes["gene_name"].isin(primary_genes)]

def filter_lncRNAs(genes: pd.DataFrame, primary_genes: list[str]) -> pd.DataFrame:
    genes = genes[(genes["feature"] == "gene")]
    return genes[genes["gene_type"] == "lncRNA"]


def _min_interval_distance(a_start, a_end, b_start, b_end):
    """
    Minimum genomic distance between intervals [a_start, a_end] and [b_start, b_end].
    Returns 0 if they overlap.
    """
    left_gap  = b_start - a_end
    right_gap = a_start - b_end
    # distance = max(0, max(left_gap, right_gap))
    return np.maximum(0, np.maximum(left_gap, right_gap))

def lncRNAs_within_window(genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, window):
    # Defensive copies & minimal selection
    g = genes[["chrom", "start", "end", "strand", "gene_name"]].copy()
    l = lncRNAs_genes[["chrom", "start", "end", "strand", "gene_name"]].copy()

    # Normalize types & sort
    for df in (g, l):
        df["start"] = df["start"].astype(np.int64)
        df["end"]   = df["end"].astype(np.int64)
        # Make chromosome categorical for fast grouping if many contigs
        df["chrom"] = df["chrom"].astype("category")

    # Precompute expanded gene windows (strand-agnostic)
    g["w_start"] = (g["start"] - window).clip(lower=0)
    g["w_end"]   = g["end"] + window

    pairs = []  # collect candidate pairs per chromosome

    # Process per chromosome to avoid cross-chrom checks
    chroms = np.intersect1d(g["chrom"].cat.categories.astype(str), l["chrom"].cat.categories.astype(str))

    for c in chroms:
        g_c = g[g["chrom"].astype(str) == c].sort_values("w_start", kind="mergesort").reset_index(drop=True)
        l_c = l[l["chrom"].astype(str) == c].sort_values("start",  kind="mergesort").reset_index(drop=True)

        if g_c.empty or l_c.empty:
            continue

        # Arrays for lncRNA positions
        l_start = l_c["start"].to_numpy()
        l_end   = l_c["end"].to_numpy()

        # For each gene window, quickly cut candidate lncRNAs by start <= w_end (via searchsorted)
        w_start = g_c["w_start"].to_numpy()
        w_end   = g_c["w_end"].to_numpy()

        # searchsorted gives the index of first lncRNA with start > w_end[i]
        # So candidate lncRNAs are 0 .. idx_end[i]-1, then we filter with l_end >= w_start[i]
        idx_end = np.searchsorted(l_start, w_end, side="right")

        for i in range(g_c.shape[0]):
            upto = idx_end[i]
            if upto == 0:
                continue
            # candidates by start
            cand_idx = np.arange(upto, dtype=np.int64)

            # filter by end >= w_start[i]
            mask = l_end[cand_idx] >= w_start[i]
            if not np.any(mask):
                continue

            sel = cand_idx[mask]
            if sel.size == 0:
                continue

            # compute minimal distance between gene interval and lncRNA interval
            dist = _min_interval_distance(
                g_c.at[i, "start"], g_c.at[i, "end"],
                l_c["start"].to_numpy()[sel], l_c["end"].to_numpy()[sel]
            )

            # build rows
            sub = pd.DataFrame({
                "chrom": c,
                "gene_name": g_c.at[i, "gene_name"],
                "gene_strand": g_c.at[i, "strand"],
                "gene_start": g_c.at[i, "start"],
                "gene_end": g_c.at[i, "end"],
                "lncRNA_name": l_c["gene_name"].to_numpy()[sel],
                "lncRNA_start": l_c["start"].to_numpy()[sel],
                "lncRNA_end": l_c["end"].to_numpy()[sel],
                "lncRNA_strand": l_c["strand"].to_numpy()[sel],
                "min_distance_bp": dist
            })
            pairs.append(sub)

    if len(pairs) == 0:
        pairs_df = pd.DataFrame(columns=[
            "chrom","gene_name","gene_start","gene_end",
            "lncRNA_name","lncRNA_start","lncRNA_end","min_distance_bp"
        ])
    else:
        pairs_df = pd.concat(pairs, ignore_index=True)

    # Deduplicate (some lncRNAs may be captured multiple times if identical coords appear)
    pairs_df = pairs_df.drop_duplicates(
        ["chrom", "gene_name", "lncRNA_name", "gene_start", "gene_end", "lncRNA_start", "lncRNA_end"]
    ).reset_index(drop=True)

    # ---- Per-gene aggregation: list of lncRNAs within 1Mb
    lnc_per_gene = (
        pairs_df.groupby("gene_name", sort=False)["lncRNA_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("lncRNAs_within_1Mb")
        .reset_index()
    )
    genes_out = genes.copy()
    genes_out = genes_out.merge(lnc_per_gene, on="gene_name", how="left")
    genes_out["lncRNAs_within_1Mb"] = genes_out["lncRNAs_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    # ---- Per-lncRNA aggregation: list of genes within 1Mb (nice to have)
    genes_per_lnc = (
        pairs_df.groupby("lncRNA_name", sort=False)["gene_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("genes_within_1Mb")
        .reset_index()
    )
    lncRNAs_out = lncRNAs_genes.copy()
    lncRNAs_out = lncRNAs_out.merge(genes_per_lnc, left_on="gene_name", right_on="lncRNA_name", how="left")
    lncRNAs_out.drop(columns=["lncRNA_name"], inplace=True)
    lncRNAs_out["genes_within_1Mb"] = lncRNAs_out["genes_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    # lncRNAs_out["genes_within_1Mb_list"] = lncRNAs_out["genes_within_1Mb"].apply(safe_parse_list)
    # lncRNAs_out["num_genes"] = lncRNAs_out["genes_within_1Mb_list"].apply(len)

    lncRNAs_out["num_genes"] = lncRNAs_out["genes_within_1Mb"].apply(len)
    lncRNAs_out = lncRNAs_out[lncRNAs_out["num_genes"] > 0]

    # lncRNAs_out.drop(columns=["genes_within_1Mb"], inplace=True)
    # lncRNAs_out.rename(columns={"genes_within_1Mb_list": "genes_within_1Mb"}, inplace=True)

    return pairs_df, genes_out, lncRNAs_out

def save_lncRNA_matching(working_dir: str, pairs_df: pd.DataFrame, genes_with_lists: pd.DataFrame, lncRNAs_with_lists: pd.DataFrame, window: int):
    lncRNA_matching_dir = os.path.join(working_dir, "lncRNA_matching")
    os.makedirs(lncRNA_matching_dir, exist_ok=True)
    pairs_df.to_csv(os.path.join(lncRNA_matching_dir, f"genes_lncRNAs_{window}bp_distances.csv"), index=False)
    genes_with_lists = genes_with_lists[["chrom","start","end","gene_name","gene_id","lncRNAs_within_1Mb"]]
    genes_with_lists.to_csv(os.path.join(lncRNA_matching_dir, f"genes_with_lncRNAs_{window}bp.csv"), index=False)
    lncRNAs_with_lists = lncRNAs_with_lists[["chrom","start","end","gene_name","gene_id","genes_within_1Mb", "num_genes", "strand"]]
    lncRNAs_with_lists.to_csv(os.path.join(lncRNA_matching_dir, f"lncRNAs_with_genes_{window}bp.csv"), index=False)
    lncRNAs_with_lists["gene_name"].to_csv(os.path.join(lncRNA_matching_dir, f"lncRNAs_names_with_genes_{window}bp_names.csv"), index=False)

def match_genes_lncRNAs(working_dir: str, genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, window) -> list[str]:
    pairs_df, genes_with_lists, lncRNAs_with_lists = lncRNAs_within_window(genes, lncRNAs_genes, window)
    save_lncRNA_matching(working_dir, pairs_df, genes_with_lists, lncRNAs_with_lists, window)
    return lncRNAs_with_lists["gene_name"].to_list()


def extract_primary_class(s: str) -> str:
    if not isinstance(s, str):
        return ""
    # First token is usually the core class (PLS, pELS, dELS, CTCF-only, DNase-H3K4me3, etc.)
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return parts[0] if parts else ""



def handle_cell_line(
    cell_line_dir_path: str,
    elem_focus: pd.DataFrame,
    cell_line: str,
    wanted_signals: list[str],
    elem_id_col_in_bed: int = 3,
) -> pd.DataFrame:
    """
    For each element (row) creates a dict in column `cell_line`, e.g.:

    elem_focus[cell_line].iloc[i] == {
        f"in_{cell_line}": bool,
        "CTCF": float or None,
        "DNase": float or None,
        "ATAC": float or None,
        "H3K27ac": float or None,
        "H3K27me3": float or None,
    }

    The signal names come from `wanted_signals`.
    """
    df = elem_focus.copy()

    # -------- 1. Get the set of cCRE IDs for that cell line --------
    bed_files = [f for f in os.listdir(cell_line_dir_path) if f.endswith(".bed")]
    if len(bed_files) != 1:
        raise ValueError(
            f"Expected exactly one .bed file in {cell_line_dir_path}, found {len(bed_files)}"
        )

    bed_path = os.path.join(cell_line_dir_path, bed_files[0])
    bed = pd.read_csv(bed_path, sep="\t", header=None)
    ccre_ids = set(bed.iloc[:, elem_id_col_in_bed].tolist())

    in_key = f"in_{cell_line}"
    df[in_key] = df["ENCODE_id"].isin(ccre_ids)

    # -------- 2. Load signals into temporary columns --------
    # We'll build a dict: signal_name -> column_name in df
    signal_cols = {}  # e.g., {"CTCF": "sig_CTCF", "DNase": "sig_DNase", ...}

    for file in os.listdir(cell_line_dir_path):
        base, ext = os.path.splitext(file)  # "CTCF", ".txt" etc.
        if base in wanted_signals:
            signal_path = os.path.join(cell_line_dir_path, file)
            signal_df = pd.read_csv(signal_path, sep="\t", header=None)

            # Assume first column = elem_id, second column = signal value
            # Build a mapping elem_id -> value
            id_col = signal_df.columns[0]
            val_col = signal_df.columns[1]
            signal_map = signal_df.set_index(id_col)[val_col]

            # Temporary column to hold raw signal values
            col_name = f"__{cell_line}_{base}__"
            df[col_name] = df["ENCODE_id"].map(signal_map).astype("float64")  # or skip astype
            signal_cols[base] = col_name

    # -------- 3. Build dict per row in one go --------
    def make_dict(row):
        d = {in_key: bool(row[in_key])}
        for sig_name in wanted_signals:
            if sig_name in signal_cols:
                val = row[signal_cols[sig_name]]
                # Use None instead of NaN if you prefer empty
                d[sig_name] = None if pd.isna(val) else val
            else:
                # No file for this signal in this cell line directory
                d[sig_name] = None
        return d

    df[cell_line] = df.apply(make_dict, axis=1)

    # Optionally drop the temporary helper columns
    df.drop(columns=[in_key] + list(signal_cols.values()), inplace=True)

    return df




COLS_RAW = [
    "cCRE ID",
    "Gene ID",
    "Common Gene Name",
    "Gene Type",
    "Assay Type",
    "Experiment ID",
    "Biosample",
    "Score",
    "P-value",
]


def build_links_table(
    zip_path: str,
    inner_file: str,
    gene_list: Iterable[str],
    panel_biosamples: Optional[Iterable[str]] = None,
    breast_biosamples: Optional[Iterable[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    intact_pvalue_strong: Optional[float] = None,
    global_biosamples: Optional[Iterable[str]] = None,
    chunksize: int = 300_000,
) -> pd.DataFrame:
    """
    Build a table with one row per (cCRE, gene) link and:

    - gene_type
    - one column per panel_biosample: dict {assay_type: {"score", "p_value", "strength"}}
    - conservation_global: dict {assay_type: {"n_biosamples", "n_strong", "n_weak"}}
    - conservation_breast: same but restricted to breast_biosamples

    Parameters
    ----------
    zip_path : str
        Path to Human-Gene-Links.zip.
    inner_file : str
        Name of the TXT file inside the ZIP (e.g. "V4-hg38.Gene-Links.3D-Chromatin.txt").
    gene_list : iterable of str
        Gene symbols to keep (matched against "Common Gene Name").
    panel_biosamples : iterable of str, optional
        Biosamples that will become columns with dicts (one column per biosample).
        If None, no per-biosample columns are added.
    breast_biosamples : iterable of str, optional
        Biosamples used for conservation_breast; if None, uses panel_biosamples.
    weak_quantile : float
        Lower quantile used to define "weak" links (score >= weak_quantile and < strong_quantile).
    strong_quantile : float
        Upper quantile used to define "strong" links (score >= strong_quantile).
    intact_pvalue_strong : float, optional
        If not None, then for assay_type == "Intact-HiC", a link is only "strong" if
        score >= strong_quantile AND p_value <= intact_pvalue_strong.
    global_biosamples : iterable of str, optional
        Which biosamples to use when computing conservation_global.
        If None, all biosamples present in the data are used.
    chunksize : int
        Number of rows per chunk when streaming the big TXT file.

    Returns
    -------
    links : pd.DataFrame
        Table with one row per (cCRE, gene) and nested dict columns as described.
    """
    gene_set = set(gene_list)

    # Panel biosamples
    panel_biosamples = list(panel_biosamples) if panel_biosamples is not None else []
    # Breast biosamples
    if breast_biosamples is None:
        breast_biosample_set = set(panel_biosamples)
    else:
        breast_biosample_set = set(breast_biosamples)

    # 1. Stream file & extract rows for selected genes (all biosamples)
    chunks = []
    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_file) as f:
            for chunk in pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=COLS_RAW,
                chunksize=chunksize,
                low_memory=False,
            ):
                chunk["Common Gene Name"] = (
                    chunk["Common Gene Name"].astype(str).str.strip()
                )
                filt = chunk["Common Gene Name"].isin(gene_set)
                if filt.any():
                    chunks.append(chunk.loc[filt])

    if not chunks:
        raise ValueError("No rows found for the selected genes.")

    df_primary = pd.concat(chunks, ignore_index=True)
    print("Total rows extracted for selected genes:", len(df_primary))

    # 2. Clean column names, numeric types
    df_primary = df_primary.rename(
        columns={
            "cCRE ID": "cCRE_id",
            "Gene ID": "gene_id",
            "Common Gene Name": "gene_name",
            "Gene Type": "gene_type",
            "Assay Type": "assay_type",
            "Experiment ID": "experiment_id",
            "Biosample": "biosample",
            "Score": "score",
            "P-value": "p_value",
        }
    )

    df_primary["score"] = pd.to_numeric(df_primary["score"], errors="coerce")
    df_primary["p_value"] = pd.to_numeric(df_primary["p_value"], errors="coerce")

    # 3. Per-assay quantiles (computed AFTER filtering to selected genes)
    quantiles = (
        df_primary.groupby("assay_type")["score"]
        .quantile([weak_quantile, strong_quantile])
        .unstack()
        .rename(columns={weak_quantile: "q_weak", strong_quantile: "q_strong"})
    )

    print("\nPer-assay score quantiles on selected genes:")
    print(quantiles)

    df_primary = df_primary.merge(
        quantiles, left_on="assay_type", right_index=True, how="left"
    )

    # 4. Define strength per row: strong / weak / none (per assay_type)
    df_primary["is_strong"] = df_primary["score"] >= df_primary["q_strong"]
    df_primary["is_weak"] = (df_primary["score"] >= df_primary["q_weak"]) & (
        df_primary["score"] < df_primary["q_strong"]
    )

    # Optional: refine Intact-HiC with p_value
    if intact_pvalue_strong is not None:
        mask_intact = df_primary["assay_type"].eq("Intact-HiC")
        df_primary.loc[mask_intact, "is_strong"] &= (
            df_primary.loc[mask_intact, "p_value"] <= intact_pvalue_strong
        )

    def classify_strength(row) -> str:
        if row["is_strong"]:
            return "strong"
        elif row["is_weak"]:
            return "weak"
        else:
            return "none"

    df_primary["strength"] = df_primary.apply(classify_strength, axis=1)

    # 5. Collapse to per-link/per-biosample/per-assay summary
    link_cols = ["cCRE_id", "gene_id", "gene_name"]
    group_link_bio_assay = link_cols + ["biosample", "assay_type"]

    per_link_bio_assay = (
        df_primary.groupby(group_link_bio_assay)
        .agg(
            max_score=("score", "max"),
            min_p_value=("p_value", "min"),
            any_strong=("is_strong", "max"),
            any_weak=("is_weak", "max"),
            gene_type=("gene_type", "first"),  # usually constant per gene
        )
        .reset_index()
    )

    def classify_strength_from_flags(row) -> str:
        if row["any_strong"]:
            return "strong"
        elif row["any_weak"]:
            return "weak"
        else:
            return "none"

    per_link_bio_assay["strength"] = per_link_bio_assay.apply(
        classify_strength_from_flags, axis=1
    )

    # 6. Base "links" table: one row per (cCRE, gene)
    links = (
        per_link_bio_assay[link_cols + ["gene_type"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # 7. Add one column per panel biosample: dict per assay
    def make_biosample_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        """
        df_sub: rows for a single (link, biosample) across assay_types
        returns: { assay_type: {"score": ..., "p_value": ..., "strength": ...}, ... }
        """
        out: Dict[str, Dict[str, Any]] = {}
        for _, r in df_sub.iterrows():
            out[r["assay_type"]] = {
                "score": float(r["max_score"]) if pd.notna(r["max_score"]) else None,
                "p_value": float(r["min_p_value"])
                if pd.notna(r["min_p_value"])
                else None,
                "strength": r["strength"],
            }
        return out

    if panel_biosamples:
        for bio in panel_biosamples:
            df_bio = per_link_bio_assay[per_link_bio_assay["biosample"] == bio]
            if df_bio.empty:
                # create column, will fill with {} later
                links[bio] = [{} for _ in range(len(links))]
                continue

            bio_series = df_bio.groupby(link_cols).apply(make_biosample_dict)
            links = links.merge(
                bio_series.rename(bio).reset_index(), on=link_cols, how="left"
            )

        # Replace NaN in panel biosample columns with {}
        for bio in panel_biosamples:
            if bio in links.columns:
                links[bio] = links[bio].apply(
                    lambda x: x if isinstance(x, dict) else {}
                )

    # 8. Build conservation dicts: global and breast, stratified by assay_type
    # Apply global biosample filter if provided
    if global_biosamples is not None:
        global_bio_set = set(global_biosamples)
        per_link_bio_assay_global = per_link_bio_assay[
            per_link_bio_assay["biosample"].isin(global_bio_set)
        ]
    else:
        per_link_bio_assay_global = per_link_bio_assay

    def make_cons_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, int]]:
        d: Dict[str, Dict[str, int]] = {}
        for _, r in df_sub.iterrows():
            d[r["assay_type"]] = {
                "n_biosamples": int(r["n_biosamples"]),
                "n_strong": int(r["n_strong"]),
                "n_weak": int(r["n_weak"]),
            }
        return d

    # GLOBAL conservation
    cons_global = (
        per_link_bio_assay_global.groupby(link_cols + ["assay_type"])
        .agg(
            n_biosamples=("biosample", "nunique"),
            n_strong=("any_strong", "sum"),
            n_weak=("any_weak", "sum"),
        )
        .reset_index()
    )

    cons_global_series = (
        cons_global.groupby(link_cols)
        .apply(make_cons_dict)
        .rename("conservation_global")
        .reset_index()
    )

    links = links.merge(cons_global_series, on=link_cols, how="left")

    # BREAST conservation
    per_link_bio_assay_breast = per_link_bio_assay[
        per_link_bio_assay["biosample"].isin(breast_biosample_set)
    ]

    if not per_link_bio_assay_breast.empty:
        cons_breast = (
            per_link_bio_assay_breast.groupby(link_cols + ["assay_type"])
            .agg(
                n_biosamples=("biosample", "nunique"),
                n_strong=("any_strong", "sum"),
                n_weak=("any_weak", "sum"),
            )
            .reset_index()
        )

        cons_breast_series = (
            cons_breast.groupby(link_cols)
            .apply(make_cons_dict)
            .rename("conservation_breast")
            .reset_index()
        )

        links = links.merge(cons_breast_series, on=link_cols, how="left")
    else:
        links["conservation_breast"] = [{} for _ in range(len(links))]

    # Ensure dict columns are {} instead of NaN if missing
    for col in ["conservation_global", "conservation_breast"]:
        if col in links.columns:
            links[col] = links[col].apply(lambda x: x if isinstance(x, dict) else {})

    return links




def default_panel_biosamples() -> List[str]:
    """Default breast / mammary biosamples that will get their own dict columns."""
    return [
        "MCF-7",
        "MCF_10A",
        "Breast Mammary Tissue",
        "breast_epithelium_tissue_female_adult_(53_years)",
        "mammary_epithelial_cell_female_adult_(19_years)",
    ]


def build_element_table(links: pd.DataFrame) -> pd.DataFrame:
    """
    Condense a link-level table (one row per cCRE-gene) into an
    element-level table (one row per cCRE_id) with a single column 'gene_links'.

    'gene_links' is a dict:
        {
          gene_name: {
              # all original columns from the links dataframe
              # except for 'cCRE_id' and 'gene_name'
              # (these are used as keys / implied by the outer structure)
              "gene_id": ...,
              "gene_type": ...,
              "<biosample_1>": {...},
              "<biosample_2>": {...},
              "conservation_global": {...},
              "conservation_breast": {...},
              ...
          },
          ...
        }

    Parameters
    ----------
    links : pd.DataFrame
        The link-level table produced by your build_links_table()
        function, with at least:
        ['cCRE_id', 'gene_id', 'gene_name', 'gene_type', ...].

    Returns
    -------
    elements : pd.DataFrame
        DataFrame with columns:
          - 'cCRE_id'
          - 'gene_links' (dict as described above)
    """

    # sanity check
    required_cols = {"cCRE_id", "gene_id", "gene_name", "gene_type"}
    missing = required_cols - set(links.columns)
    if missing:
        raise ValueError(f"links is missing required columns: {missing}")

    # function to build the dict per element
    def make_gene_links_for_element(df_sub: pd.DataFrame):
        """
        df_sub: all rows in 'links' corresponding to a single cCRE_id.
        Returns:
            {
              gene_name_1: { all columns except cCRE_id and gene_name },
              gene_name_2: { ... },
              ...
            }
        """
        out = {}
        for _, row in df_sub.iterrows():
            gene_name = row["gene_name"]
            # convert full row to dict
            row_dict = row.to_dict()

            # remove redundant/outer keys
            row_dict.pop("cCRE_id", None)
            row_dict.pop("gene_name", None)

            # keep gene_id, gene_type and all per-biosample & conservation fields
            out[gene_name] = row_dict
        return out

    # group by cCRE_id and build the gene_links dict
    gene_links_series = (
        links
        .groupby("cCRE_id", sort=False)
        .apply(make_gene_links_for_element)
        .rename("gene_links")
    )

    # turn into a DataFrame: one row per cCRE_id
    elements = gene_links_series.reset_index()

    # no NaN here; every cCRE_id has at least one gene, so every 'gene_links'
    # entry is a non-empty dict
    return elements


def build_promoter_columns(genes: pd.DataFrame, tss_up: int = 2000, tss_down: int = 500) -> pd.DataFrame:
    genes["prom_start"] = genes.apply(lambda x: max(x["start"] - tss_up, 0) if x["strand"] == "+" else x["end"] + tss_up, axis=1) # to add chromosme end to the other side
    genes["prom_end"] = genes.apply(lambda x: min(x["start"] + tss_down, x["end"]) if x["strand"] == "+" else max(x["start"] - tss_down, 0), axis=1)
    return genes





def match_genes_regulatory_elements(working_dir: str, 
    genes: pd.DataFrame,  
    gene_type: str, 
    regulatory_elements: pd.DataFrame, 
    cell_lines_dir_path, 
    cell_lines_wanted, 
    wanted_signals: list[str],
    gene_links_zip_path: str,
    gene_links_inner_file: str,
    elem_id_col_in_bed: int = 3, 
    window: int = 1_000_000) -> pd.DataFrame:
    # -----------------------------
    # Config
    # -----------------------------
    BIN = 100_000     # coarse prefilter bin size

    # Tiers (right-closed). Keep your labels and build matching edges.
    TIER_EDGES  = [0, 100_000, 250_000, 500_000, 1_000_000]                # bp
    TIER_LABELS = ["0–100kb", "100–250kb", "250–500kb", "500–1000kb"]      # 4 labels
    
    genes["start"]  = genes["start"].astype(np.int64)
    genes["end"]    = genes["end"].astype(np.int64)
    regulatory_elements["start"] = regulatory_elements["start"].astype(np.int64)
    regulatory_elements["end"]   = regulatory_elements["end"].astype(np.int64)
    genes  = _as_categorical(genes,  ["chrom", "gene_name", "strand"])
    regulatory_elements["raw_type"] = regulatory_elements["type"].astype(str)
    regulatory_elements["type"] = regulatory_elements["raw_type"].apply(extract_primary_class)
    regulatory_elements = _as_categorical(regulatory_elements, ["chrom", "type"])

    # -----------------------------
    # Gene TSS and ±W window (strand-aware; DOES NOT omit '-' genes)
    # -----------------------------
    genes["tss"]       = np.where(genes["strand"].astype(str) == "+", genes["start"], genes["end"]).astype(np.int64)
    genes["win_start"] = (genes["tss"] - window).clip(lower=0).astype(np.int64)
    genes["win_end"]   = (genes["tss"] + window).astype(np.int64)

    # -----------------------------
    # Element metadata & ID
    # -----------------------------
    regulatory_elements["center"] = ((regulatory_elements["start"] + regulatory_elements["end"]) // 2).astype(np.int64)

    # Prefer cCRE_id if present and non-empty; else ENCODE_id
    cCRE = regulatory_elements["cCRE_id"]   if "cCRE_id"   in regulatory_elements.columns else pd.Series("", index=regulatory_elements.index)
    eID  = regulatory_elements["ENCODE_id"] if "ENCODE_id" in regulatory_elements.columns else pd.Series("", index=regulatory_elements.index)
    elem_key = np.where(cCRE.astype(str).str.len() > 0, cCRE.astype(str), eID.astype(str))
    regulatory_elements["elem_id"] = pd.Series(elem_key, index=regulatory_elements.index).astype("category")
    

    # -----------------------------
    # Coarse prefilter via bins
    # -----------------------------
    genes_bins  = add_bins(genes,  "win_start", "win_end", BIN)
    encode_bins = add_bins(regulatory_elements, "start",     "end",     BIN)

    gb = genes_bins.explode("bins")
    eb = encode_bins.explode("bins")

    pref = (
        gb[["gene_name","chrom","win_start","win_end","tss","bins"]]
        .merge(
            eb[["chrom","start","end","center","elem_id", "ENCODE_id", "type", "raw_type", "bins"]],
            on=["chrom","bins"], how="inner", copy=False
        )
    )

    # Tighten with real window overlap
    pref = pref[(pref["end"] >= pref["win_start"]) & (pref["start"] <= pref["win_end"])].copy()

    # -----------------------------
    # Exact distance from TSS to element interval
    # distance = 0 if TSS in [start, end]; else min(|tss-start|, |tss-end|)
    # -----------------------------
    tss = pref["tss"].to_numpy(np.int64)
    s   = pref["start"].to_numpy(np.int64)
    e   = pref["end"].to_numpy(np.int64)

    inside = (tss >= s) & (tss <= e)
    dist   = np.where(inside, 0, np.minimum(np.abs(tss - s), np.abs(tss - e)))
    pref["dist_to_tss"] = dist.astype(np.int64)

    # Keep within ±W by exact distance (explicit)
    pref = pref[pref["dist_to_tss"] <= window].copy()

    # -----------------------------
    # Tier labels (fix edges vs labels mismatch)
    # -----------------------------
    bins_for_cut = _make_cut_edges_for_labels(TIER_EDGES, include_zero_bin=True)
    pref["tier"] = pd.cut(
        pref["dist_to_tss"],
        bins=bins_for_cut,
        labels=TIER_LABELS,
        include_lowest=True,
        right=True,
        ordered=True
    )

    # -----------------------------
    # 1) Long table of all gene–element pairs (exact)
    # -----------------------------
    pair_df = pref[[
        "gene_name","elem_id","ENCODE_id","type", "raw_type", "chrom","start","end","center","tss","dist_to_tss","tier"
    ]].copy()

    # -----------------------------
    # 2) Element-focus table (tiered gene lists + min distance + exact "gene:dist")
    # -----------------------------
    agg_genes = (pair_df
        .groupby(["elem_id", "ENCODE_id", "type", "raw_type", "chrom","start","end","tier"], observed=True)["gene_name"]
        .apply(lambda s: ",".join(sorted(pd.unique(s))))
        .reset_index()
    )

    elem_focus = (agg_genes
        .pivot(index=["elem_id", "ENCODE_id", "type", "raw_type", "chrom","start","end"], columns="tier", values="gene_name")
        .reset_index()
    )

    # Cast tier columns to object BEFORE fillna to avoid categorical fill errors
    tier_cols = [c for c in elem_focus.columns if c in TIER_LABELS]
    for c in tier_cols:
        elem_focus[c] = elem_focus[c].astype(object)
    elem_focus[tier_cols] = elem_focus[tier_cols].fillna("")

    # Min distance to any gene
    elem_focus = elem_focus.merge(
        pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().rename("min_dist_to_any_gene"),
        on="elem_id", how="left"
    )

    # Exact "gene:dist" compact string (sorted by distance)
    _exact = (pair_df
        .sort_values(["elem_id","dist_to_tss","gene_name"])
        .groupby("elem_id", observed=True)
        .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))
        .rename("genes_by_exact_dist")
        .reset_index()
    )
    elem_focus = elem_focus.merge(_exact, on="elem_id", how="left")
    if gene_type == "coding":
        for cell_line in os.listdir(cell_lines_dir_path):
            if cell_line in cell_lines_wanted:
                cell_line_dir_path = os.path.join(cell_lines_dir_path, cell_line)
                elem_focus = handle_cell_line(cell_line_dir_path, elem_focus, cell_line, wanted_signals, elem_id_col_in_bed=elem_id_col_in_bed)
            
    

        genes = genes["gene_name"].tolist()
        panel_bios = default_panel_biosamples()

        # Example: strong = top 5%, weak = top 10–5%, Intact-HiC strong p <= 1e-6
        links_df = build_links_table(
            zip_path=gene_links_zip_path,
            inner_file=gene_links_inner_file,
            gene_list=genes,
            panel_biosamples=panel_bios,
            breast_biosamples=panel_bios,  # can be different if you want
            weak_quantile=0.90,
            strong_quantile=0.95,
            intact_pvalue_strong=1e-6,
            global_biosamples=None,  # use all biosamples for global conservation
            chunksize=300_000,
        )
        links_df_condense = build_element_table(links_df)
        elem_focus = elem_focus.merge(links_df_condense, left_on="elem_id", right_on="cCRE_id", how="left")

    # -----------------------------
    # 3) Gene summary table: counts and IDs per (type, tier), wide format
    # -----------------------------
    # Counts
    cnt = (pair_df
        .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
        .nunique()
        .rename("count")
        .reset_index())

    cnt_wide = (cnt
        .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | count")
        .pivot(index="gene_name", columns="col", values="count")
        .reset_index()
    )

    # IDs
    ids = (pair_df
        .groupby(["gene_name","type","tier"], observed=True)["elem_id"]
        .apply(lambda s: ",".join(sorted(pd.unique(s))))
        .rename("ids")
        .reset_index())

    ids_wide = (ids
        .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | ids")
        .pivot(index="gene_name", columns="col", values="ids")
        .reset_index()
    )

    # Merge, then fill numeric vs string separately (avoid categorical fill errors)
    gene_summary = cnt_wide.merge(ids_wide, on="gene_name", how="outer")

    num_cols = [c for c in gene_summary.columns if c.endswith(" | count")]
    id_cols  = [c for c in gene_summary.columns if c.endswith(" | ids")]

    # ensure proper dtypes before filling
    for c in num_cols:
        gene_summary[c] = pd.to_numeric(gene_summary[c], errors="coerce")
    for c in id_cols:
        gene_summary[c] = gene_summary[c].astype(object)

    gene_summary[num_cols] = gene_summary[num_cols].fillna(0).astype(int)
    gene_summary[id_cols]  = gene_summary[id_cols].fillna("")

    # Order columns
    gene_summary = gene_summary[["gene_name"] + [c for c in gene_summary.columns if c != "gene_name"]]

    # -----------------------------
    # 4) Distance matrix (genes × elements, min exact distance)
    # -----------------------------
    dist_matrix = (pair_df
                .groupby(["gene_name","elem_id"], observed=True)["dist_to_tss"]
                .min()
                .unstack("elem_id")
                .astype("float64"))

    # Optional: order columns by element’s global min distance
    elem_order = pair_df.groupby("elem_id", observed=True)["dist_to_tss"].min().sort_values().index
    dist_matrix = dist_matrix.reindex(columns=elem_order)
    regulatory_elements_matching_dir = os.path.join(working_dir, "regulatory_elements_matching")

    os.makedirs(regulatory_elements_matching_dir, exist_ok=True)
    if gene_type == "coding":
        pair_df.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_to_elements.csv"), index=False)
        elem_focus.to_csv(os.path.join(regulatory_elements_matching_dir, "regulatory_element_focus.csv"), index=False)
        links_df.to_csv(os.path.join(regulatory_elements_matching_dir, "links_genes_df.csv"), index=False)
        gene_summary.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_focus.csv"), index=False)
        dist_matrix.to_csv(os.path.join(regulatory_elements_matching_dir, "distance_matrix.csv"), index=False)
    else:
        regulatory_elements_matching_dir = os.path.join(regulatory_elements_matching_dir, f"{gene_type}")
        os.makedirs(regulatory_elements_matching_dir, exist_ok=True)
        pair_df.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_to_elements.csv"), index=False)
        elem_focus.to_csv(os.path.join(regulatory_elements_matching_dir, "regulatory_element_focus.csv"), index=False)
        # links_df.to_csv(os.path.join(regulatory_elements_matching_dir, "links_lncRNAs_df.csv"), index=False)
        gene_summary.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_focus.csv"), index=False)
        dist_matrix.to_csv(os.path.join(regulatory_elements_matching_dir, "distance_matrix.csv"), index=False)


def get_miRNAs_targetscan(genes, top_n=200, weight_threshold=-0.2):
    targetscan_df = pd.read_csv(
        "data/miRNA/Predicted_Targets_Context_Scores.default_predictions.txt",
        sep="\t"
    )
    df = targetscan_df[(targetscan_df["Gene Tax ID"] == 9606) & (targetscan_df["Gene ID"].str.split('.').str[0].isin(genes["gene_id"].str.split('.').str[0]))]

    # aggregate per (gene, miRNA)
    gene_miRNA_best = (
        df.groupby(["Gene ID", "Gene Symbol", "miRNA"], as_index=False)
        .agg({
            "weighted context++ score": "min",                 # best score
            "UTR_start": lambda x: list(x),                    # list of starts
            "UTR end": lambda x: list(x),                      # list of ends
        })
    )

    # add number of binding sites
    gene_miRNA_best["num_sites"] = gene_miRNA_best["UTR_start"].apply(len)

    top_hits = (
        gene_miRNA_best
        .sort_values(["Gene ID","weighted context++ score"])
        .groupby("Gene ID")
        .head(top_n)       # top 3 miRNAs per gene
    )

    # Optional threshold for strong repression
    top_hits = top_hits[top_hits["weighted context++ score"] <= weight_threshold]

    miRNA_dir = os.path.join(working_dir, "miRNA")
    os.makedirs(miRNA_dir,exist_ok=True)

    gene_miRNA_best.to_csv(os.path.join(miRNA_dir, "all_miRNAs_targetscan.csv"), index=False)
    top_hits.to_csv(os.path.join(miRNA_dir, f"top_{top_n}_threshold_{weight_threshold}_targetscan.csv"), index=False)


def create_genes_bed(genes):
    # Convert to BED coordinates
    df_bed = genes.copy()
    df_bed["start"] = df_bed["start"] - 1 # bed files are 0 based

    # Keep only relevant columns
    df_bed = df_bed[["chrom", "start", "end", "gene_name", "strand"]]

    # Write BED (tab-separated, no header)
    df_bed.to_csv("data/apm_genes.bed", sep="\t", header=False, index=False)





def process_genes_general(working_dir: str,
    genes_file: str,
    lncRNAs_file: str,
    regulatory_elements_path: str, 
    primary_genes: list[str], 
    cell_lines_dir_path: str, 
    cell_lines_wanted, 
    wanted_signals: list[str], 
    gene_links_zip_path: str,
    gene_links_inner_file: str,
    elem_id_col_in_cell_line_bed: int = 3,
    window: int = 1_000_000) -> pd.DataFrame:

    
    genes_path = os.path.join(working_dir, genes_file)
    lncRNAs_path = os.path.join(working_dir, lncRNAs_file)
    genes = pd.read_csv(genes_path)
    lncRNAs_genes = pd.read_csv(lncRNAs_path)
    regulatory_elements = pd.read_csv(regulatory_elements_path)
    paths = [genes_path, lncRNAs_path, regulatory_elements_path]


    genes, lncRNAs_genes, regulatory_elements, was_changed = harmonize_chrom(genes, lncRNAs_genes, regulatory_elements)
    for i, df in enumerate([genes, lncRNAs_genes, regulatory_elements]):
        if was_changed[i]:
            df.to_csv(paths[i], index=False)
    
    genes_only = filter_selected_genes(genes, primary_genes)
    genes_only = build_promoter_columns(genes_only, tss_up=2000, tss_down=500)
    lncRNAs_genes_only = filter_lncRNAs(lncRNAs_genes, primary_genes)
    lncRNAs_genes_only = build_promoter_columns(lncRNAs_genes_only, tss_up=2000, tss_down=500)
    matched_lncRNAs_lst = match_genes_lncRNAs(working_dir, genes_only, lncRNAs_genes_only, window)
    lncRNAs_genes_only = lncRNAs_genes_only[lncRNAs_genes_only["gene_name"].isin(matched_lncRNAs_lst)]


    match_genes_regulatory_elements(working_dir, genes_only,"coding", regulatory_elements, cell_lines_dir_path, cell_lines_wanted, 
    wanted_signals, gene_links_zip_path, gene_links_inner_file, elem_id_col_in_cell_line_bed, window)

    match_genes_regulatory_elements(working_dir, lncRNAs_genes_only, "lncRNA", regulatory_elements, cell_lines_dir_path, cell_lines_wanted, 
    wanted_signals, gene_links_zip_path, gene_links_inner_file, elem_id_col_in_cell_line_bed, window)

    get_miRNAs_targetscan(genes,top_n=200, weight_threshold=-0.2)
    create_genes_bed(genes)
    genes[genes["gene_name"].isin(primary_genes)].to_csv(f"{working_dir}/primary_genes_all_features.csv", index=False)
    genes[(genes["gene_name"].isin(primary_genes)) & (genes["feature"] == "gene")].to_csv(f"{working_dir}/primary_genes_only.csv", index=False)
    genes[genes["gene_name"].isin(lncRNAs_genes_only["gene_name"])].to_csv(f"{working_dir}/lncRNAs_genes_all_features.csv", index=False)


working_dir = r"C:\Users\stavz\Desktop\masters\APM\test"
genes_file = os.path.join(working_dir, "gencode.v49.annotation.gtf.csv")
lncRNAs_file = os.path.join(working_dir, "lncRNAs_all_features.csv")
regulatory_elements_path = os.path.join(working_dir, "GRCh38-cCREs.csv")
gene_links_zip_path = r"C:\Users\stavz\Downloads\Human-Gene-Links.zip"
gene_links_inner_file = "V4-hg38.Gene-Links.3D-Chromatin.txt"
cell_lines_dir_path = r"C:\Users\stavz\Desktop\masters\APM\data\cell_lines cCREs data"
cell_lines_wanted = ["MCF7", "HMEC1"]
wanted_signals = ["H3K27ac",  "H3K4me3", "CTCF", "DNase"]
elem_id_col_in_cell_line_bed = 3

primary_genes = [
    "PSMB8", "PSMB9", "PSMB10",
    "PSME1", "PSME2", "PSME4",
    "TAP1", "TAP2", "TAPBP", "TAPBPL",
    "CALR", "PDIA3", "PDIA6",
    "B2M",
    "HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G",
    "MICA", "MICB",
    "ULBP1", "ULBP2", "ULBP3", "RAET1E", "RAET1G", "RAET1L", # ULBP4, ULBP5, ULBP6
    "PVR", "NECTIN2", "NCR3LG1",
    "SOCS1", "SOCS3",
    "NFKBIA", "TNFAIP3",
    "STAT1", "STAT3",
    "JAK1", "JAK2",
    "IRF1", "IRF2",
    "NLRC5",
    "CD274",          # PD-L1
    "PDCD1LG2",       # PD-L2
    "IFNG",
    "IFNGR1", "IFNGR2",
    "ADAM10", "ADAM17",
    "TGFB1",
    "IL6", "IL6R", "IL6ST",
    "PIAS1", "PIAS3",
    "PTPN2", "PTPN11",
    "EGFR",
    "SRC",
    "SMAD3",
    "EP300", "CREBBP", "CCL5", "CXCL9", "CXCL10", "CXCL11"]
window = 1_000_000

process_genes_general(working_dir, genes_file, lncRNAs_file, regulatory_elements_path, 
primary_genes, cell_lines_dir_path, cell_lines_wanted, wanted_signals, gene_links_zip_path, gene_links_inner_file, 
elem_id_col_in_cell_line_bed, window)




    


C:\Users\stavz\AppData\Local\Temp\ipykernel_27580\3458551182.py:1032: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  genes = pd.read_csv(genes_path)


0
1
2


C:\Users\stavz\AppData\Local\Temp\ipykernel_27580\3458551182.py:854: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\stavz\\Desktop\\masters\\APM\\data\\cell_lines cCREs data'

In [ ]:
import pandas as pd
s = pd.read_csv(r"C:\Users\stavz\Desktop\masters\APM\test\regulatory_elements_matching\links_genes_df.csv")


In [3]:
import pandas as pd
import os
working_dir = r"C:\Users\stavz\Desktop\masters\APM\test"

regulatory_elements_path = os.path.join(working_dir, "GRCh38-cCREs.csv")
reg = pd.read_csv(regulatory_elements_path)


    


In [ ]:
######################################################################### NEW UNTESTED CODE #########################################################################


import pandas as pd
import numpy as np
import ast
import os 
import zipfile
from typing import List, Optional, Iterable, Dict, Any

def _normalize_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.str.replace(r"^(?!(chr))", "chrom", regex=True)

def add_bins(df: pd.DataFrame, start_col: str, end_col: str, bin_size) -> pd.DataFrame:
    """Coarse binning for fast prefilter; exact distances still computed later."""
    out = df.copy()
    out["bin_start"] = (out[start_col] // bin_size).astype(np.int64)
    out["bin_end"]   = (out[end_col]   // bin_size).astype(np.int64)
    out["bins"] = [list(range(s, e + 1)) for s, e in zip(out["bin_start"], out["bin_end"])]
    return out

def _make_cut_edges_for_labels(base_edges, include_zero_bin=True):
    """
    Make edges so that len(labels) == len(edges)-1.
    With base_edges=[0,100k,250k,500k,1M] and 4 labels, we need edges[-0.1,100k,250k,500k,1M].
    """
    if include_zero_bin:
        return [-0.1] + base_edges[1:]
    else:
        return base_edges

def _as_categorical(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

def safe_parse_list(x):
    if pd.isna(x) or x in ["", "[]"]:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            # fall back: split comma-separated strings
            return [g for g in x.split(",") if g.strip() != ""]
    return []


def harmonize_chrom(genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, regulatory_elements: pd.DataFrame):
    was_changed = [False, False, False]
    for i, df in enumerate([genes, lncRNAs_genes, regulatory_elements]):
        print(i)
        if "seqname" in df.columns:
            df.rename(columns={"seqname": "chrom"}, inplace=True)
            was_changed[i] = True
        elif isinstance(df["chrom"].iloc[0], int):
            df["chrom"] = _normalize_chr(df["chrom"])
            was_changed[i] = True
    return genes, lncRNAs_genes, regulatory_elements, was_changed


def filter_selected_genes(genes: pd.DataFrame, primary_genes: list[str]) -> pd.DataFrame:
    genes = genes[genes["feature"] == "gene"]
    return genes[genes["gene_name"].isin(primary_genes)]

def filter_lncRNAs(genes: pd.DataFrame, primary_genes: list[str]) -> pd.DataFrame:
    genes = genes[(genes["feature"] == "gene")]
    return genes[genes["gene_type"] == "lncRNA"]


def _min_interval_distance(a_start, a_end, b_start, b_end):
    """
    Minimum genomic distance between intervals [a_start, a_end] and [b_start, b_end].
    Returns 0 if they overlap.
    """
    left_gap  = b_start - a_end
    right_gap = a_start - b_end
    # distance = max(0, max(left_gap, right_gap))
    return np.maximum(0, np.maximum(left_gap, right_gap))

def lncRNAs_within_window(genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, window):
    # Defensive copies & minimal selection
    g = genes[["chrom", "start", "end", "strand", "gene_name"]].copy()
    l = lncRNAs_genes[["chrom", "start", "end", "strand", "gene_name"]].copy()

    # Normalize types & sort
    for df in (g, l):
        df["start"] = df["start"].astype(np.int64)
        df["end"]   = df["end"].astype(np.int64)
        # Make chromosome categorical for fast grouping if many contigs
        df["chrom"] = df["chrom"].astype("category")

    # Precompute expanded gene windows (strand-agnostic)
    g["w_start"] = (g["start"] - window).clip(lower=0)
    g["w_end"]   = g["end"] + window

    pairs = []  # collect candidate pairs per chromosome

    # Process per chromosome to avoid cross-chrom checks
    chroms = np.intersect1d(g["chrom"].cat.categories.astype(str), l["chrom"].cat.categories.astype(str))

    for c in chroms:
        g_c = g[g["chrom"].astype(str) == c].sort_values("w_start", kind="mergesort").reset_index(drop=True)
        l_c = l[l["chrom"].astype(str) == c].sort_values("start",  kind="mergesort").reset_index(drop=True)

        if g_c.empty or l_c.empty:
            continue

        # Arrays for lncRNA positions
        l_start = l_c["start"].to_numpy()
        l_end   = l_c["end"].to_numpy()

        # For each gene window, quickly cut candidate lncRNAs by start <= w_end (via searchsorted)
        w_start = g_c["w_start"].to_numpy()
        w_end   = g_c["w_end"].to_numpy()

        # searchsorted gives the index of first lncRNA with start > w_end[i]
        # So candidate lncRNAs are 0 .. idx_end[i]-1, then we filter with l_end >= w_start[i]
        idx_end = np.searchsorted(l_start, w_end, side="right")

        for i in range(g_c.shape[0]):
            upto = idx_end[i]
            if upto == 0:
                continue
            # candidates by start
            cand_idx = np.arange(upto, dtype=np.int64)

            # filter by end >= w_start[i]
            mask = l_end[cand_idx] >= w_start[i]
            if not np.any(mask):
                continue

            sel = cand_idx[mask]
            if sel.size == 0:
                continue

            # compute minimal distance between gene interval and lncRNA interval
            dist = _min_interval_distance(
                g_c.at[i, "start"], g_c.at[i, "end"],
                l_c["start"].to_numpy()[sel], l_c["end"].to_numpy()[sel]
            )

            # build rows
            sub = pd.DataFrame({
                "chrom": c,
                "gene_name": g_c.at[i, "gene_name"],
                "gene_strand": g_c.at[i, "strand"],
                "gene_start": g_c.at[i, "start"],
                "gene_end": g_c.at[i, "end"],
                "lncRNA_name": l_c["gene_name"].to_numpy()[sel],
                "lncRNA_start": l_c["start"].to_numpy()[sel],
                "lncRNA_end": l_c["end"].to_numpy()[sel],
                "lncRNA_strand": l_c["strand"].to_numpy()[sel],
                "min_distance_bp": dist
            })
            pairs.append(sub)

    if len(pairs) == 0:
        pairs_df = pd.DataFrame(columns=[
            "chrom","gene_name","gene_start","gene_end",
            "lncRNA_name","lncRNA_start","lncRNA_end","min_distance_bp"
        ])
    else:
        pairs_df = pd.concat(pairs, ignore_index=True)

    # Deduplicate (some lncRNAs may be captured multiple times if identical coords appear)
    pairs_df = pairs_df.drop_duplicates(
        ["chrom", "gene_name", "lncRNA_name", "gene_start", "gene_end", "lncRNA_start", "lncRNA_end"]
    ).reset_index(drop=True)

    # ---- Per-gene aggregation: list of lncRNAs within 1Mb
    lnc_per_gene = (
        pairs_df.groupby("gene_name", sort=False)["lncRNA_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("lncRNAs_within_1Mb")
        .reset_index()
    )
    genes_out = genes.copy()
    genes_out = genes_out.merge(lnc_per_gene, on="gene_name", how="left")
    genes_out["lncRNAs_within_1Mb"] = genes_out["lncRNAs_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    # ---- Per-lncRNA aggregation: list of genes within 1Mb (nice to have)
    genes_per_lnc = (
        pairs_df.groupby("lncRNA_name", sort=False)["gene_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("genes_within_1Mb")
        .reset_index()
    )
    lncRNAs_out = lncRNAs_genes.copy()
    lncRNAs_out = lncRNAs_out.merge(genes_per_lnc, left_on="gene_name", right_on="lncRNA_name", how="left")
    lncRNAs_out.drop(columns=["lncRNA_name"], inplace=True)
    lncRNAs_out["genes_within_1Mb"] = lncRNAs_out["genes_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    # lncRNAs_out["genes_within_1Mb_list"] = lncRNAs_out["genes_within_1Mb"].apply(safe_parse_list)
    # lncRNAs_out["num_genes"] = lncRNAs_out["genes_within_1Mb_list"].apply(len)

    lncRNAs_out["num_genes"] = lncRNAs_out["genes_within_1Mb"].apply(len)
    lncRNAs_out = lncRNAs_out[lncRNAs_out["num_genes"] > 0]

    # lncRNAs_out.drop(columns=["genes_within_1Mb"], inplace=True)
    # lncRNAs_out.rename(columns={"genes_within_1Mb_list": "genes_within_1Mb"}, inplace=True)

    return pairs_df, genes_out, lncRNAs_out

def save_lncRNA_matching(working_dir: str, pairs_df: pd.DataFrame, genes_with_lists: pd.DataFrame, lncRNAs_with_lists: pd.DataFrame, window: int):
    lncRNA_matching_dir = os.path.join(working_dir, "lncRNA_matching")
    os.makedirs(lncRNA_matching_dir, exist_ok=True)
    pairs_df.to_csv(os.path.join(lncRNA_matching_dir, f"genes_lncRNAs_{window}bp_distances.csv"), index=False)
    genes_with_lists = genes_with_lists[["chrom","start","end","gene_name","gene_id","lncRNAs_within_1Mb"]]
    genes_with_lists.to_csv(os.path.join(lncRNA_matching_dir, f"genes_with_lncRNAs_{window}bp.csv"), index=False)
    lncRNAs_with_lists = lncRNAs_with_lists[["chrom","start","end","gene_name","gene_id","genes_within_1Mb", "num_genes", "strand"]]
    lncRNAs_with_lists.to_csv(os.path.join(lncRNA_matching_dir, f"lncRNAs_with_genes_{window}bp.csv"), index=False)
    lncRNAs_with_lists["gene_name"].to_csv(os.path.join(lncRNA_matching_dir, f"lncRNAs_names_with_genes_{window}bp_names.csv"), index=False)

def match_genes_lncRNAs(working_dir: str, genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, window) -> list[str]:
    pairs_df, genes_with_lists, lncRNAs_with_lists = lncRNAs_within_window(genes, lncRNAs_genes, window)
    save_lncRNA_matching(working_dir, pairs_df, genes_with_lists, lncRNAs_with_lists, window)
    return lncRNAs_with_lists["gene_name"].to_list()


def extract_primary_class(s: str) -> str:
    if not isinstance(s, str):
        return ""
    # First token is usually the core class (PLS, pELS, dELS, CTCF-only, DNase-H3K4me3, etc.)
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return parts[0] if parts else ""



def handle_cell_line(
    cell_line_dir_path: str,
    elem_focus: pd.DataFrame,
    cell_line: str,
    wanted_signals: list[str],
    cCRE_id_col_in_bed: int = 3,
) -> pd.DataFrame:
    """
    For each element (row) creates a dict in column `cell_line`, e.g.:

    elem_focus[cell_line].iloc[i] == {
        f"in_{cell_line}": bool,
        "CTCF": float or None,
        "DNase": float or None,
        "ATAC": float or None,
        "H3K27ac": float or None,
        "H3K27me3": float or None,
    }

    The signal names come from `wanted_signals`.
    """
    df = elem_focus.copy()

    # -------- 1. Get the set of cCRE IDs for that cell line --------
    bed_files = [f for f in os.listdir(cell_line_dir_path) if f.endswith(".bed")]
    if len(bed_files) != 1:
        raise ValueError(
            f"Expected exactly one .bed file in {cell_line_dir_path}, found {len(bed_files)}"
        )

    bed_path = os.path.join(cell_line_dir_path, bed_files[0])
    bed = pd.read_csv(bed_path, sep="\t", header=None)
    ccre_ids = set(bed.iloc[:, cCRE_id_col_in_bed].tolist())

    in_key = f"in_{cell_line}"
    df[in_key] = df["ENCODE_id"].isin(ccre_ids)

    # -------- 2. Load signals into temporary columns --------
    # We'll build a dict: signal_name -> column_name in df
    signal_cols = {}  # e.g., {"CTCF": "sig_CTCF", "DNase": "sig_DNase", ...}

    for file in os.listdir(cell_line_dir_path):
        base, ext = os.path.splitext(file)  # "CTCF", ".txt" etc.
        if base in wanted_signals:
            signal_path = os.path.join(cell_line_dir_path, file)
            signal_df = pd.read_csv(signal_path, sep="\t", header=None)

            # Assume first column = cCRE_id, second column = signal value
            # Build a mapping cCRE_id -> value
            id_col = signal_df.columns[0]
            val_col = signal_df.columns[1]
            signal_map = signal_df.set_index(id_col)[val_col]

            # Temporary column to hold raw signal values
            col_name = f"__{cell_line}_{base}__"
            df[col_name] = df["ENCODE_id"].map(signal_map).astype("float64")  # or skip astype
            signal_cols[base] = col_name

    # -------- 3. Build dict per row in one go --------
    def make_dict(row):
        d = {in_key: bool(row[in_key])}
        for sig_name in wanted_signals:
            if sig_name in signal_cols:
                val = row[signal_cols[sig_name]]
                # Use None instead of NaN if you prefer empty
                d[sig_name] = None if pd.isna(val) else val
            else:
                # No file for this signal in this cell line directory
                d[sig_name] = None
        return d

    df[cell_line] = df.apply(make_dict, axis=1)

    # Optionally drop the temporary helper columns
    df.drop(columns=[in_key] + list(signal_cols.values()), inplace=True)

    return df




COLS_RAW = [
    "cCRE ID",
    "Gene ID",
    "Common Gene Name",
    "Gene Type",
    "Assay Type",
    "Experiment ID",
    "Biosample",
    "Score",
    "P-value",
]


def build_links_table(
    zip_path: str,
    inner_file: str,
    gene_list: Iterable[str],
    panel_biosamples: Optional[Iterable[str]] = None,
    breast_biosamples: Optional[Iterable[str]] = None,
    weak_quantile: float = 0.90,
    strong_quantile: float = 0.95,
    intact_pvalue_strong: Optional[float] = None,
    global_biosamples: Optional[Iterable[str]] = None,
    chunksize: int = 300_000,
) -> pd.DataFrame:
    """
    Build a table with one row per (cCRE, gene) link and:

    - gene_type
    - one column per panel_biosample: dict {assay_type: {"score", "p_value", "strength"}}
    - conservation_global: dict {assay_type: {"n_biosamples", "n_strong", "n_weak"}}
    - conservation_breast: same but restricted to breast_biosamples

    Parameters
    ----------
    zip_path : str
        Path to Human-Gene-Links.zip.
    inner_file : str
        Name of the TXT file inside the ZIP (e.g. "V4-hg38.Gene-Links.3D-Chromatin.txt").
    gene_list : iterable of str
        Gene symbols to keep (matched against "Common Gene Name").
    panel_biosamples : iterable of str, optional
        Biosamples that will become columns with dicts (one column per biosample).
        If None, no per-biosample columns are added.
    breast_biosamples : iterable of str, optional
        Biosamples used for conservation_breast; if None, uses panel_biosamples.
    weak_quantile : float
        Lower quantile used to define "weak" links (score >= weak_quantile and < strong_quantile).
    strong_quantile : float
        Upper quantile used to define "strong" links (score >= strong_quantile).
    intact_pvalue_strong : float, optional
        If not None, then for assay_type == "Intact-HiC", a link is only "strong" if
        score >= strong_quantile AND p_value <= intact_pvalue_strong.
    global_biosamples : iterable of str, optional
        Which biosamples to use when computing conservation_global.
        If None, all biosamples present in the data are used.
    chunksize : int
        Number of rows per chunk when streaming the big TXT file.

    Returns
    -------
    links : pd.DataFrame
        Table with one row per (cCRE, gene) and nested dict columns as described.
    """
    gene_set = set(gene_list)

    # Panel biosamples
    panel_biosamples = list(panel_biosamples) if panel_biosamples is not None else []
    # Breast biosamples
    if breast_biosamples is None:
        breast_biosample_set = set(panel_biosamples)
    else:
        breast_biosample_set = set(breast_biosamples)

    # 1. Stream file & extract rows for selected genes (all biosamples)
    chunks = []
    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_file) as f:
            for chunk in pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=COLS_RAW,
                chunksize=chunksize,
                low_memory=False,
            ):
                chunk["Common Gene Name"] = (
                    chunk["Common Gene Name"].astype(str).str.strip()
                )
                filt = chunk["Common Gene Name"].isin(gene_set)
                if filt.any():
                    chunks.append(chunk.loc[filt])

    if not chunks:
        raise ValueError("No rows found for the selected genes.")

    df_primary = pd.concat(chunks, ignore_index=True)
    print("Total rows extracted for selected genes:", len(df_primary))

    # 2. Clean column names, numeric types
    df_primary = df_primary.rename(
        columns={
            "cCRE ID": "cCRE_id",
            "Gene ID": "gene_id",
            "Common Gene Name": "gene_name",
            "Gene Type": "gene_type",
            "Assay Type": "assay_type",
            "Experiment ID": "experiment_id",
            "Biosample": "biosample",
            "Score": "score",
            "P-value": "p_value",
        }
    )

    df_primary["score"] = pd.to_numeric(df_primary["score"], errors="coerce")
    df_primary["p_value"] = pd.to_numeric(df_primary["p_value"], errors="coerce")

    # 3. Per-assay quantiles (computed AFTER filtering to selected genes)
    quantiles = (
        df_primary.groupby("assay_type")["score"]
        .quantile([weak_quantile, strong_quantile])
        .unstack()
        .rename(columns={weak_quantile: "q_weak", strong_quantile: "q_strong"})
    )

    print("\nPer-assay score quantiles on selected genes:")
    print(quantiles)

    df_primary = df_primary.merge(
        quantiles, left_on="assay_type", right_index=True, how="left"
    )

    # 4. Define strength per row: strong / weak / none (per assay_type)
    df_primary["is_strong"] = df_primary["score"] >= df_primary["q_strong"]
    df_primary["is_weak"] = (df_primary["score"] >= df_primary["q_weak"]) & (
        df_primary["score"] < df_primary["q_strong"]
    )

    # Optional: refine Intact-HiC with p_value
    if intact_pvalue_strong is not None:
        mask_intact = df_primary["assay_type"].eq("Intact-HiC")
        df_primary.loc[mask_intact, "is_strong"] &= (
            df_primary.loc[mask_intact, "p_value"] <= intact_pvalue_strong
        )

    def classify_strength(row) -> str:
        if row["is_strong"]:
            return "strong"
        elif row["is_weak"]:
            return "weak"
        else:
            return "none"

    df_primary["strength"] = df_primary.apply(classify_strength, axis=1)

    # 5. Collapse to per-link/per-biosample/per-assay summary
    link_cols = ["cCRE_id", "gene_id", "gene_name"]
    group_link_bio_assay = link_cols + ["biosample", "assay_type"]

    per_link_bio_assay = (
        df_primary.groupby(group_link_bio_assay)
        .agg(
            max_score=("score", "max"),
            min_p_value=("p_value", "min"),
            any_strong=("is_strong", "max"),
            any_weak=("is_weak", "max"),
            gene_type=("gene_type", "first"),  # usually constant per gene
        )
        .reset_index()
    )

    def classify_strength_from_flags(row) -> str:
        if row["any_strong"]:
            return "strong"
        elif row["any_weak"]:
            return "weak"
        else:
            return "none"

    per_link_bio_assay["strength"] = per_link_bio_assay.apply(
        classify_strength_from_flags, axis=1
    )

    # 6. Base "links" table: one row per (cCRE, gene)
    links = (
        per_link_bio_assay[link_cols + ["gene_type"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # 7. Add one column per panel biosample: dict per assay
    def make_biosample_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        """
        df_sub: rows for a single (link, biosample) across assay_types
        returns: { assay_type: {"score": ..., "p_value": ..., "strength": ...}, ... }
        """
        out: Dict[str, Dict[str, Any]] = {}
        for _, r in df_sub.iterrows():
            out[r["assay_type"]] = {
                "score": float(r["max_score"]) if pd.notna(r["max_score"]) else None,
                "p_value": float(r["min_p_value"])
                if pd.notna(r["min_p_value"])
                else None,
                "strength": r["strength"],
            }
        return out

    if panel_biosamples:
        for bio in panel_biosamples:
            df_bio = per_link_bio_assay[per_link_bio_assay["biosample"] == bio]
            if df_bio.empty:
                # create column, will fill with {} later
                links[bio] = [{} for _ in range(len(links))]
                continue

            bio_series = df_bio.groupby(link_cols).apply(make_biosample_dict)
            links = links.merge(
                bio_series.rename(bio).reset_index(), on=link_cols, how="left"
            )

        # Replace NaN in panel biosample columns with {}
        for bio in panel_biosamples:
            if bio in links.columns:
                links[bio] = links[bio].apply(
                    lambda x: x if isinstance(x, dict) else {}
                )

    # 8. Build conservation dicts: global and breast, stratified by assay_type
    # Apply global biosample filter if provided
    if global_biosamples is not None:
        global_bio_set = set(global_biosamples)
        per_link_bio_assay_global = per_link_bio_assay[
            per_link_bio_assay["biosample"].isin(global_bio_set)
        ]
    else:
        per_link_bio_assay_global = per_link_bio_assay

    def make_cons_dict(df_sub: pd.DataFrame) -> Dict[str, Dict[str, int]]:
        d: Dict[str, Dict[str, int]] = {}
        for _, r in df_sub.iterrows():
            d[r["assay_type"]] = {
                "n_biosamples": int(r["n_biosamples"]),
                "n_strong": int(r["n_strong"]),
                "n_weak": int(r["n_weak"]),
            }
        return d

    # GLOBAL conservation
    cons_global = (
        per_link_bio_assay_global.groupby(link_cols + ["assay_type"])
        .agg(
            n_biosamples=("biosample", "nunique"),
            n_strong=("any_strong", "sum"),
            n_weak=("any_weak", "sum"),
        )
        .reset_index()
    )

    cons_global_series = (
        cons_global.groupby(link_cols)
        .apply(make_cons_dict)
        .rename("conservation_global")
        .reset_index()
    )

    links = links.merge(cons_global_series, on=link_cols, how="left")

    # BREAST conservation
    per_link_bio_assay_breast = per_link_bio_assay[
        per_link_bio_assay["biosample"].isin(breast_biosample_set)
    ]

    if not per_link_bio_assay_breast.empty:
        cons_breast = (
            per_link_bio_assay_breast.groupby(link_cols + ["assay_type"])
            .agg(
                n_biosamples=("biosample", "nunique"),
                n_strong=("any_strong", "sum"),
                n_weak=("any_weak", "sum"),
            )
            .reset_index()
        )

        cons_breast_series = (
            cons_breast.groupby(link_cols)
            .apply(make_cons_dict)
            .rename("conservation_breast")
            .reset_index()
        )

        links = links.merge(cons_breast_series, on=link_cols, how="left")
    else:
        links["conservation_breast"] = [{} for _ in range(len(links))]

    # Ensure dict columns are {} instead of NaN if missing
    for col in ["conservation_global", "conservation_breast"]:
        if col in links.columns:
            links[col] = links[col].apply(lambda x: x if isinstance(x, dict) else {})

    return links




def default_panel_biosamples() -> List[str]:
    """Default breast / mammary biosamples that will get their own dict columns."""
    return [
        "MCF-7",
        "MCF_10A",
        "Breast Mammary Tissue",
        "breast_epithelium_tissue_female_adult_(53_years)",
        "mammary_epithelial_cell_female_adult_(19_years)",
    ]


def build_element_table(links: pd.DataFrame) -> pd.DataFrame:
    """
    Condense a link-level table (one row per cCRE-gene) into an
    element-level table (one row per cCRE_id) with a single column 'gene_links'.

    'gene_links' is a dict:
        {
          gene_name: {
              # all original columns from the links dataframe
              # except for 'cCRE_id' and 'gene_name'
              # (these are used as keys / implied by the outer structure)
              "gene_id": ...,
              "gene_type": ...,
              "<biosample_1>": {...},
              "<biosample_2>": {...},
              "conservation_global": {...},
              "conservation_breast": {...},
              ...
          },
          ...
        }

    Parameters
    ----------
    links : pd.DataFrame
        The link-level table produced by your build_links_table()
        function, with at least:
        ['cCRE_id', 'gene_id', 'gene_name', 'gene_type', ...].

    Returns
    -------
    elements : pd.DataFrame
        DataFrame with columns:
          - 'cCRE_id'
          - 'gene_links' (dict as described above)
    """

    # sanity check
    required_cols = {"cCRE_id", "gene_id", "gene_name", "gene_type"}
    missing = required_cols - set(links.columns)
    if missing:
        raise ValueError(f"links is missing required columns: {missing}")

    # function to build the dict per element
    def make_gene_links_for_element(df_sub: pd.DataFrame):
        """
        df_sub: all rows in 'links' corresponding to a single cCRE_id.
        Returns:
            {
              gene_name_1: { all columns except cCRE_id and gene_name },
              gene_name_2: { ... },
              ...
            }
        """
        out = {}
        for _, row in df_sub.iterrows():
            gene_name = row["gene_name"]
            # convert full row to dict
            row_dict = row.to_dict()

            # remove redundant/outer keys
            row_dict.pop("cCRE_id", None)
            row_dict.pop("gene_name", None)

            # keep gene_id, gene_type and all per-biosample & conservation fields
            out[gene_name] = row_dict
        return out

    # group by cCRE_id and build the gene_links dict
    gene_links_series = (
        links
        .groupby("cCRE_id", sort=False)
        .apply(make_gene_links_for_element)
        .rename("gene_links")
    )

    # turn into a DataFrame: one row per cCRE_id
    elements = gene_links_series.reset_index()

    # no NaN here; every cCRE_id has at least one gene, so every 'gene_links'
    # entry is a non-empty dict
    return elements


def build_promoter_columns(genes: pd.DataFrame, tss_up: int = 2000, tss_down: int = 500) -> pd.DataFrame:
    genes["prom_start"] = genes.apply(lambda x: max(x["start"] - tss_up, 0) if x["strand"] == "+" else x["end"] + tss_up, axis=1) # to add chromosme end to the other side
    genes["prom_end"] = genes.apply(lambda x: min(x["start"] + tss_down, x["end"]) if x["strand"] == "+" else max(x["start"] - tss_down, 0), axis=1)
    return genes





def match_genes_regulatory_elements(working_dir: str, 
    genes: pd.DataFrame,  
    gene_type: str, 
    regulatory_elements: pd.DataFrame, 
    cell_lines_dir_path, 
    cell_lines_wanted, 
    wanted_signals: list[str],
    gene_links_zip_path: str,
    gene_links_inner_file: str,
    cCRE_id_col_in_bed: int = 3, 
    window: int = 1_000_000) -> pd.DataFrame:
    # -----------------------------
    # Config
    # -----------------------------
    BIN = 100_000     # coarse prefilter bin size

    # Tiers (right-closed). Keep your labels and build matching edges.
    TIER_EDGES  = [0, 100_000, 250_000, 500_000, 1_000_000]                # bp
    TIER_LABELS = ["0–100kb", "100–250kb", "250–500kb", "500–1000kb"]      # 4 labels
    
    genes["start"]  = genes["start"].astype(np.int64)
    genes["end"]    = genes["end"].astype(np.int64)
    regulatory_elements["start"] = regulatory_elements["start"].astype(np.int64)
    regulatory_elements["end"]   = regulatory_elements["end"].astype(np.int64)
    genes  = _as_categorical(genes,  ["chrom", "gene_name", "strand"])
    regulatory_elements["raw_type"] = regulatory_elements["type"].astype(str)
    regulatory_elements["type"] = regulatory_elements["raw_type"].apply(extract_primary_class)
    regulatory_elements = _as_categorical(regulatory_elements, ["chrom", "type"])

    # -----------------------------
    # Gene TSS and ±W window (strand-aware; DOES NOT omit '-' genes)
    # -----------------------------
    genes["tss"]       = np.where(genes["strand"].astype(str) == "+", genes["start"], genes["end"]).astype(np.int64)
    genes["win_start"] = (genes["tss"] - window).clip(lower=0).astype(np.int64)
    genes["win_end"]   = (genes["tss"] + window).astype(np.int64)

    # -----------------------------
    # Element metadata & ID
    # -----------------------------
    regulatory_elements["center"] = ((regulatory_elements["start"] + regulatory_elements["end"]) // 2).astype(np.int64)

    # Prefer cCRE_id if present and non-empty; else ENCODE_id
    # cCRE = regulatory_elements["cCRE_id"]   if "cCRE_id"   in regulatory_elements.columns else pd.Series("", index=regulatory_elements.index)
    # eID  = regulatory_elements["ENCODE_id"] if "ENCODE_id" in regulatory_elements.columns else pd.Series("", index=regulatory_elements.index)
    # elem_key = np.where(cCRE.astype(str).str.len() > 0, cCRE.astype(str), eID.astype(str))
    # regulatory_elements["cCRE_id"] = pd.Series(elem_key, index=regulatory_elements.index).astype("category")
    

    # -----------------------------
    # Coarse prefilter via bins
    # -----------------------------
    genes_bins  = add_bins(genes,  "win_start", "win_end", BIN)
    encode_bins = add_bins(regulatory_elements, "start",     "end",     BIN)

    gb = genes_bins.explode("bins")
    eb = encode_bins.explode("bins")

    pref = (
        gb[["gene_name","chrom","win_start","win_end","tss","bins"]]
        .merge(
            eb[["chrom","start","end","center","cCRE_id", "ENCODE_id", "type", "raw_type", "bins"]],
            on=["chrom","bins"], how="inner", copy=False
        )
    )

    # Tighten with real window overlap
    pref = pref[(pref["end"] >= pref["win_start"]) & (pref["start"] <= pref["win_end"])].copy()

    # -----------------------------
    # Exact distance from TSS to element interval
    # distance = 0 if TSS in [start, end]; else min(|tss-start|, |tss-end|)
    # -----------------------------
    tss = pref["tss"].to_numpy(np.int64)
    s   = pref["start"].to_numpy(np.int64)
    e   = pref["end"].to_numpy(np.int64)

    inside = (tss >= s) & (tss <= e)
    dist   = np.where(inside, 0, np.minimum(np.abs(tss - s), np.abs(tss - e)))
    pref["dist_to_tss"] = dist.astype(np.int64)

    # Keep within ±W by exact distance (explicit)
    pref = pref[pref["dist_to_tss"] <= window].copy()

    # -----------------------------
    # Tier labels (fix edges vs labels mismatch)
    # -----------------------------
    bins_for_cut = _make_cut_edges_for_labels(TIER_EDGES, include_zero_bin=True)
    pref["tier"] = pd.cut(
        pref["dist_to_tss"],
        bins=bins_for_cut,
        labels=TIER_LABELS,
        include_lowest=True,
        right=True,
        ordered=True
    )

    # -----------------------------
    # 1) Long table of all gene–element pairs (exact)
    # -----------------------------
    pair_df = pref[[
        "gene_name","cCRE_id","ENCODE_id","type", "raw_type", "chrom","start","end","center","tss","dist_to_tss","tier"
    ]].copy()

    # -----------------------------
    # 2) Element-focus table (tiered gene lists + min distance + exact "gene:dist")
    # -----------------------------
    agg_genes = (pair_df
        .groupby(["cCRE_id", "ENCODE_id", "type", "raw_type", "chrom","start","end","tier"], observed=True)["gene_name"]
        .apply(lambda s: ",".join(sorted(pd.unique(s))))
        .reset_index()
    )

    elem_focus = (agg_genes
        .pivot(index=["cCRE_id", "ENCODE_id", "type", "raw_type", "chrom","start","end"], columns="tier", values="gene_name")
        .reset_index()
    )

    # Cast tier columns to object BEFORE fillna to avoid categorical fill errors
    tier_cols = [c for c in elem_focus.columns if c in TIER_LABELS]
    for c in tier_cols:
        elem_focus[c] = elem_focus[c].astype(object)
    elem_focus[tier_cols] = elem_focus[tier_cols].fillna("")

    # Min distance to any gene
    elem_focus = elem_focus.merge(
        pair_df.groupby("cCRE_id", observed=True)["dist_to_tss"].min().rename("min_dist_to_any_gene"),
        on="cCRE_id", how="left"
    )

    # Exact "gene:dist" compact string (sorted by distance)
    _exact = (pair_df
        .sort_values(["cCRE_id","dist_to_tss","gene_name"])
        .groupby("cCRE_id", observed=True)
        .apply(lambda d: ",".join(f"{g}:{int(dd)}" for g, dd in zip(d["gene_name"], d["dist_to_tss"])))
        .rename("genes_by_exact_dist")
        .reset_index()
    )
    elem_focus = elem_focus.merge(_exact, on="cCRE_id", how="left")
    if gene_type == "coding":
        for cell_line in os.listdir(cell_lines_dir_path):
            if cell_line in cell_lines_wanted:
                cell_line_dir_path = os.path.join(cell_lines_dir_path, cell_line)
                elem_focus = handle_cell_line(cell_line_dir_path, elem_focus, cell_line, wanted_signals, cCRE_id_col_in_bed=cCRE_id_col_in_bed)
            
    

        genes = genes["gene_name"].tolist()
        panel_bios = default_panel_biosamples()

        # Example: strong = top 5%, weak = top 10–5%, Intact-HiC strong p <= 1e-6
        links_df = build_links_table(
            zip_path=gene_links_zip_path,
            inner_file=gene_links_inner_file,
            gene_list=genes,
            panel_biosamples=panel_bios,
            breast_biosamples=panel_bios,  # can be different if you want
            weak_quantile=0.90,
            strong_quantile=0.95,
            intact_pvalue_strong=1e-6,
            global_biosamples=None,  # use all biosamples for global conservation
            chunksize=300_000,
        )
        links_df_condense = build_element_table(links_df)
        elem_focus = elem_focus.merge(links_df_condense, left_on="cCRE_id", right_on="cCRE_id", how="left")

    # -----------------------------
    # 3) Gene summary table: counts and IDs per (type, tier), wide format
    # -----------------------------
    # Counts
    cnt = (pair_df
        .groupby(["gene_name","type","tier"], observed=True)["cCRE_id"]
        .nunique()
        .rename("count")
        .reset_index())

    cnt_wide = (cnt
        .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | count")
        .pivot(index="gene_name", columns="col", values="count")
        .reset_index()
    )

    # IDs
    ids = (pair_df
        .groupby(["gene_name","type","tier"], observed=True)["cCRE_id"]
        .apply(lambda s: ",".join(sorted(pd.unique(s))))
        .rename("ids")
        .reset_index())

    ids_wide = (ids
        .assign(col=lambda d: d["type"].astype(str) + " | " + d["tier"].astype(str) + " | ids")
        .pivot(index="gene_name", columns="col", values="ids")
        .reset_index()
    )

    # Merge, then fill numeric vs string separately (avoid categorical fill errors)
    gene_summary = cnt_wide.merge(ids_wide, on="gene_name", how="outer")

    num_cols = [c for c in gene_summary.columns if c.endswith(" | count")]
    id_cols  = [c for c in gene_summary.columns if c.endswith(" | ids")]

    # ensure proper dtypes before filling
    for c in num_cols:
        gene_summary[c] = pd.to_numeric(gene_summary[c], errors="coerce")
    for c in id_cols:
        gene_summary[c] = gene_summary[c].astype(object)

    gene_summary[num_cols] = gene_summary[num_cols].fillna(0).astype(int)
    gene_summary[id_cols]  = gene_summary[id_cols].fillna("")

    # Order columns
    gene_summary = gene_summary[["gene_name"] + [c for c in gene_summary.columns if c != "gene_name"]]

    # -----------------------------
    # 4) Distance matrix (genes × elements, min exact distance)
    # -----------------------------
    dist_matrix = (pair_df
                .groupby(["gene_name","cCRE_id"], observed=True)["dist_to_tss"]
                .min()
                .unstack("cCRE_id")
                .astype("float64"))

    # Optional: order columns by element’s global min distance
    elem_order = pair_df.groupby("cCRE_id", observed=True)["dist_to_tss"].min().sort_values().index
    dist_matrix = dist_matrix.reindex(columns=elem_order)
    regulatory_elements_matching_dir = os.path.join(working_dir, "regulatory_elements_matching")

    os.makedirs(regulatory_elements_matching_dir, exist_ok=True)
    if gene_type == "coding":
        pair_df.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_to_elements.csv"), index=False)
        elem_focus.to_csv(os.path.join(regulatory_elements_matching_dir, "regulatory_element_focus.csv"), index=False)
        links_df.to_csv(os.path.join(regulatory_elements_matching_dir, "links_genes_df.csv"), index=False)
        gene_summary.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_focus.csv"), index=False)
        dist_matrix.to_csv(os.path.join(regulatory_elements_matching_dir, "distance_matrix.csv"), index=False)
    else:
        regulatory_elements_matching_dir = os.path.join(regulatory_elements_matching_dir, f"{gene_type}")
        os.makedirs(regulatory_elements_matching_dir, exist_ok=True)
        pair_df.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_to_elements.csv"), index=False)
        elem_focus.to_csv(os.path.join(regulatory_elements_matching_dir, "regulatory_element_focus.csv"), index=False)
        # links_df.to_csv(os.path.join(regulatory_elements_matching_dir, "links_lncRNAs_df.csv"), index=False)
        gene_summary.to_csv(os.path.join(regulatory_elements_matching_dir, "gene_focus.csv"), index=False)
        dist_matrix.to_csv(os.path.join(regulatory_elements_matching_dir, "distance_matrix.csv"), index=False)


def get_miRNAs_targetscan(genes, top_n=200, weight_threshold=-0.2):
    targetscan_df = pd.read_csv(
        "data/miRNA/Predicted_Targets_Context_Scores.default_predictions.txt",
        sep="\t"
    )
    df = targetscan_df[(targetscan_df["Gene Tax ID"] == 9606) & (targetscan_df["Gene ID"].str.split('.').str[0].isin(genes["gene_id"].str.split('.').str[0]))]

    # aggregate per (gene, miRNA)
    gene_miRNA_best = (
        df.groupby(["Gene ID", "Gene Symbol", "miRNA"], as_index=False)
        .agg({
            "weighted context++ score": "min",                 # best score
            "UTR_start": lambda x: list(x),                    # list of starts
            "UTR end": lambda x: list(x),                      # list of ends
        })
    )

    # add number of binding sites
    gene_miRNA_best["num_sites"] = gene_miRNA_best["UTR_start"].apply(len)

    top_hits = (
        gene_miRNA_best
        .sort_values(["Gene ID","weighted context++ score"])
        .groupby("Gene ID")
        .head(top_n)       # top 3 miRNAs per gene
    )

    # Optional threshold for strong repression
    top_hits = top_hits[top_hits["weighted context++ score"] <= weight_threshold]

    miRNA_dir = os.path.join(working_dir, "miRNA")
    os.makedirs(miRNA_dir,exist_ok=True)

    gene_miRNA_best.to_csv(os.path.join(miRNA_dir, "all_miRNAs_targetscan.csv"), index=False)
    top_hits.to_csv(os.path.join(miRNA_dir, f"top_{top_n}_threshold_{weight_threshold}_targetscan.csv"), index=False)


def create_genes_bed(genes):
    # Convert to BED coordinates
    df_bed = genes.copy()
    df_bed["start"] = df_bed["start"] - 1 # bed files are 0 based

    # Keep only relevant columns
    df_bed = df_bed[["chrom", "start", "end", "gene_name", "strand"]]

    # Write BED (tab-separated, no header)
    df_bed.to_csv("data/apm_genes.bed", sep="\t", header=False, index=False)





def process_genes_general(working_dir: str,
    genes_file: str,
    lncRNAs_file: str,
    regulatory_elements_path: str, 
    primary_genes: list[str], 
    cell_lines_dir_path: str, 
    cell_lines_wanted, 
    wanted_signals: list[str], 
    gene_links_zip_path: str,
    gene_links_inner_file: str,
    cCRE_id_col_in_bed: int = 3,
    window: int = 1_000_000) -> pd.DataFrame:

    
    genes_path = os.path.join(working_dir, genes_file)
    lncRNAs_path = os.path.join(working_dir, lncRNAs_file)
    genes = pd.read_csv(genes_path)
    lncRNAs_genes = pd.read_csv(lncRNAs_path)
    regulatory_elements = pd.read_csv(regulatory_elements_path)
    paths = [genes_path, lncRNAs_path, regulatory_elements_path]


    genes, lncRNAs_genes, regulatory_elements, was_changed = harmonize_chrom(genes, lncRNAs_genes, regulatory_elements)
    for i, df in enumerate([genes, lncRNAs_genes, regulatory_elements]):
        if was_changed[i]:
            df.to_csv(paths[i], index=False)
    
    genes_only = filter_selected_genes(genes, primary_genes)
    genes_only = build_promoter_columns(genes_only, tss_up=2000, tss_down=500)
    lncRNAs_genes_only = filter_lncRNAs(lncRNAs_genes, primary_genes)
    lncRNAs_genes_only = build_promoter_columns(lncRNAs_genes_only, tss_up=2000, tss_down=500)
    matched_lncRNAs_lst = match_genes_lncRNAs(working_dir, genes_only, lncRNAs_genes_only, window)
    lncRNAs_genes_only = lncRNAs_genes_only[lncRNAs_genes_only["gene_name"].isin(matched_lncRNAs_lst)]


    match_genes_regulatory_elements(working_dir, genes_only,"coding", regulatory_elements, cell_lines_dir_path, cell_lines_wanted, 
    wanted_signals, gene_links_zip_path, gene_links_inner_file, cCRE_id_col_in_bed, window)

    match_genes_regulatory_elements(working_dir, lncRNAs_genes_only, "lncRNA", regulatory_elements, cell_lines_dir_path, cell_lines_wanted, 
    wanted_signals, gene_links_zip_path, gene_links_inner_file, cCRE_id_col_in_bed, window)

    get_miRNAs_targetscan(genes,top_n=200, weight_threshold=-0.2)
    create_genes_bed(genes)
    genes[genes["gene_name"].isin(primary_genes)].to_csv(f"{working_dir}/primary_genes_all_features.csv", index=False)
    genes[(genes["gene_name"].isin(primary_genes)) & (genes["feature"] == "gene")].to_csv(f"{working_dir}/primary_genes_only.csv", index=False)
    genes[genes["gene_name"].isin(lncRNAs_genes_only["gene_name"])].to_csv(f"{working_dir}/lncRNAs_genes_all_features.csv", index=False)


working_dir = r"C:\Users\stavz\Desktop\masters\APM\test"
genes_file = os.path.join(working_dir, "gencode.v49.annotation.gtf.csv")
lncRNAs_file = os.path.join(working_dir, "lncRNAs_all_features.csv")
regulatory_elements_path = os.path.join(working_dir, "GRCh38-cCREs.csv")
gene_links_zip_path = r"C:\Users\stavz\Downloads\Human-Gene-Links.zip"
gene_links_inner_file = "V4-hg38.Gene-Links.3D-Chromatin.txt"
cell_lines_dir_path = r"C:\Users\stavz\Desktop\masters\APM\data\cell_lines cCREs data"
cell_lines_wanted = ["MCF7", "HMEC1"]
wanted_signals = ["H3K27ac",  "H3K4me3", "CTCF", "DNase"]
cCRE_id_col_in_bed = 3

primary_genes = [
    "PSMB8", "PSMB9", "PSMB10",
    "PSME1", "PSME2", "PSME4",
    "TAP1", "TAP2", "TAPBP", "TAPBPL",
    "CALR", "PDIA3", "PDIA6",
    "B2M",
    "HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "HLA-G",
    "MICA", "MICB",
    "ULBP1", "ULBP2", "ULBP3", "RAET1E", "RAET1G", "RAET1L", # ULBP4, ULBP5, ULBP6
    "PVR", "NECTIN2", "NCR3LG1",
    "SOCS1", "SOCS3",
    "NFKBIA", "TNFAIP3",
    "STAT1", "STAT3",
    "JAK1", "JAK2",
    "IRF1", "IRF2",
    "NLRC5",
    "CD274",          # PD-L1
    "PDCD1LG2",       # PD-L2
    "IFNG",
    "IFNGR1", "IFNGR2",
    "ADAM10", "ADAM17",
    "TGFB1",
    "IL6", "IL6R", "IL6ST",
    "PIAS1", "PIAS3",
    "PTPN2", "PTPN11",
    "EGFR",
    "SRC",
    "SMAD3",
    "EP300", "CREBBP", "CCL5", "CXCL9", "CXCL10", "CXCL11"]
window = 1_000_000

process_genes_general(working_dir, genes_file, lncRNAs_file, regulatory_elements_path, 
primary_genes, cell_lines_dir_path, cell_lines_wanted, wanted_signals, gene_links_zip_path, gene_links_inner_file, 
cCRE_id_col_in_bed, window)


